# Cymatics plane → line geometry

Debug notebook for **cymatics-geometry**.

Pipeline stages:
1. Deploy points on a **100×100** square grid
2. Place **wave sources at the four corners**
3. Compute **wave interference** from corner amplitudes
4. **Displace** points in space (Z)
5. **Reconnect** displaced points into a continuous **line**

Use the sliders below to control each corner's amplitude / intensity.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

# Ensure local package is importable when the notebook is opened before `uv sync`
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from IPython.display import display, clear_output
import ipywidgets as widgets
import matplotlib.pyplot as plt

from cymatics_geometry.config import PipelineConfig
from cymatics_geometry.pipeline import run_pipeline, export_line_obj
from cymatics_geometry.visualization import show_all_stages_matplotlib

print("Ready.")

## Static run — inspect every stage

Default config uses a 100×100 grid with unequal corner amplitudes so interference is visible.

In [ ]:
config = PipelineConfig(
    grid_size=100,
    side_length=100.0,
    amplitude_sw=1.0,
    amplitude_se=0.85,
    amplitude_ne=0.55,
    amplitude_nw=0.9,
    wavelength=25.0,
    frequency=1.0,
    time=0.0,
    decay=0.0,
    line_pattern="serpentine",
)

result = run_pipeline(config, verbose=True)
result.stats

In [ ]:
# Matplotlib overview of all five stages (works in Jupyter / headless-friendly)
show_all_stages_matplotlib(result)

## Interactive corner amplitudes

Move the sliders — the field, displaced cloud, and reconnected line update live.

In [ ]:
amp_sw = widgets.FloatSlider(value=1.0, min=0.0, max=3.0, step=0.05, description="SW amp")
amp_se = widgets.FloatSlider(value=0.85, min=0.0, max=3.0, step=0.05, description="SE amp")
amp_ne = widgets.FloatSlider(value=0.55, min=0.0, max=3.0, step=0.05, description="NE amp")
amp_nw = widgets.FloatSlider(value=0.90, min=0.0, max=3.0, step=0.05, description="NW amp")
wavelength = widgets.FloatSlider(value=25.0, min=5.0, max=80.0, step=1.0, description="λ")
time = widgets.FloatSlider(value=0.0, min=0.0, max=6.28, step=0.05, description="time")
decay = widgets.FloatSlider(value=0.0, min=0.0, max=0.05, step=0.001, description="decay", readout_format=".3f")
grid_size = widgets.IntSlider(value=60, min=20, max=100, step=10, description="grid N")

out = widgets.Output()


def _rerun(_change=None) -> None:
    with out:
        clear_output(wait=True)
        cfg = PipelineConfig(
            grid_size=int(grid_size.value),
            side_length=100.0,
            amplitude_sw=float(amp_sw.value),
            amplitude_se=float(amp_se.value),
            amplitude_ne=float(amp_ne.value),
            amplitude_nw=float(amp_nw.value),
            wavelength=float(wavelength.value),
            frequency=1.0,
            time=float(time.value),
            decay=float(decay.value),
            line_pattern="serpentine",
        )
        live = run_pipeline(cfg, verbose=False)
        print(
            f"points={live.stats['point_count']}  "
            f"z∈[{live.stats['displacement_min']:.3f}, {live.stats['displacement_max']:.3f}]  "
            f"line_length={live.stats['polyline_length']:.1f}"
        )
        show_all_stages_matplotlib(live, figsize=(13, 9))


for w in (amp_sw, amp_se, amp_ne, amp_nw, wavelength, time, decay, grid_size):
    w.observe(_rerun, names="value")

controls = widgets.VBox([
    widgets.HBox([amp_sw, amp_se]),
    widgets.HBox([amp_ne, amp_nw]),
    widgets.HBox([wavelength, time]),
    widgets.HBox([decay, grid_size]),
])
display(controls, out)
_rerun()

## Optional: native PyVista windows

Uncomment one stage at a time — each opens an interactive VTK window.

In [ ]:
from cymatics_geometry.visualization import (
    show_stage_grid,
    show_stage_corners,
    show_stage_field_heatmap,
    show_stage_displaced_points,
    show_stage_line,
)

# show_stage_grid(result)
# show_stage_corners(result)
show_stage_field_heatmap(result)
# show_stage_displaced_points(result)
# show_stage_line(result)

## Export line geometry

In [ ]:
export_dir = ROOT / "exports"
path = export_line_obj(result, export_dir)
print("Wrote", path)